### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\anant\AppData\Local\Temp\ipykernel_23812\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
c:\Users\anant\OneDrive\Desktop\AI_Tutorials\RAG_PROJECT\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: attention.pdf
  ✓ Loaded 6 pages

Processing: embedding.pdf
  ✓ Loaded 19 pages

Processing: objectdetection.pdf
  ✓ Loaded 9 pages

Total documents loaded: 34


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'Apex PDFWriter', 'creator': 'PyPDF', 'creationdate': 'D:20045301135343', 'source': '..\\data\\pdf\\attention.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}, page_content='Proc.Nati.Acad.Sci.USAVol.91,pp.7398-7403,August1994\nReview\nAttention:Themechanismsofconsciousness(psychology/awareness/seection/vualsearch)\nMichaelI.Posner\nInstiteofCognitiveandDecisionScencess,UniversityofOregon,Eugene,OR97403\nABSTRACT Anumberofrecentpa-\npersandbooksdiscusstheoreticalefforts\ntowardascientificunderstandingofcon-\nsciousness.Progressinimagingnetworks\nofbrainareasactivewhenpeopleperform\nsimpletasksmayprovideausefulempir-\nicalbackgroundfordistinguishingcon-\nsciousandunconsciousinformationpro-\ncessing.Attentionalnetworksinclude\nthoseinvolvedinorientingtosensorystim-\nuli,activatingideasfrommemory,and\nmaintainingthealertstate.Thispaper\nreviewsrecentfindingsinrelationtoclas-\nsicalissuesinthestudyofat

In [4]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

## Overlap means each new chunk reuses a little bit of the previous chunk’s ending so that no important words or context get lost when you split text.

In [5]:
chunks=split_documents(all_pdf_documents)
chunks

Split 34 documents into 234 chunks

Example chunk:
Content: Proc.Nati.Acad.Sci.USAVol.91,pp.7398-7403,August1994
Review
Attention:Themechanismsofconsciousness(psychology/awareness/seection/vualsearch)
MichaelI.Posner
InstiteofCognitiveandDecisionScencess,Unive...
Metadata: {'producer': 'Apex PDFWriter', 'creator': 'PyPDF', 'creationdate': 'D:20045301135343', 'source': '..\\data\\pdf\\attention.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Apex PDFWriter', 'creator': 'PyPDF', 'creationdate': 'D:20045301135343', 'source': '..\\data\\pdf\\attention.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}, page_content='Proc.Nati.Acad.Sci.USAVol.91,pp.7398-7403,August1994\nReview\nAttention:Themechanismsofconsciousness(psychology/awareness/seection/vualsearch)\nMichaelI.Posner\nInstiteofCognitiveandDecisionScencess,UniversityofOregon,Eugene,OR97403\nABSTRACT Anumberofrecentpa-\npersandbooksdiscusstheoreticalefforts\ntowardascientificunderstandingofcon-\nsciousness.Progressinimagingnetworks\nofbrainareasactivewhenpeopleperform\nsimpletasksmayprovideausefulempir-\nicalbackgroundfordistinguishingcon-\nsciousandunconsciousinformationpro-\ncessing.Attentionalnetworksinclude\nthoseinvolvedinorientingtosensorystim-\nuli,activatingideasfrommemory,and\nmaintainingthealertstate.Thispaper\nreviewsrecentfindingsinrelationtoclas-\nsicalissuesinthestudyofat

### embedding And vectorStoreDB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2829.85it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\anant\AppData\Local\Temp\ipykernel_23812\2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### VectorStore

In [8]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


In [9]:
vectorstore = VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 234


### Existing documents in collection: 0 because we have not added any documents yet. We will add the chunks we created earlier to the vector store.
### For chunks I will extract all the text from the chunks and generate the embeddings

In [20]:
chunks

[Document(metadata={'producer': 'Apex PDFWriter', 'creator': 'PyPDF', 'creationdate': 'D:20045301135343', 'source': '..\\data\\pdf\\attention.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}, page_content='Proc.Nati.Acad.Sci.USAVol.91,pp.7398-7403,August1994\nReview\nAttention:Themechanismsofconsciousness(psychology/awareness/seection/vualsearch)\nMichaelI.Posner\nInstiteofCognitiveandDecisionScencess,UniversityofOregon,Eugene,OR97403\nABSTRACT Anumberofrecentpa-\npersandbooksdiscusstheoreticalefforts\ntowardascientificunderstandingofcon-\nsciousness.Progressinimagingnetworks\nofbrainareasactivewhenpeopleperform\nsimpletasksmayprovideausefulempir-\nicalbackgroundfordistinguishingcon-\nsciousandunconsciousinformationpro-\ncessing.Attentionalnetworksinclude\nthoseinvolvedinorientingtosensorystim-\nuli,activatingideasfrommemory,and\nmaintainingthealertstate.Thispaper\nreviewsrecentfindingsinrelationtoclas-\nsicalissuesinthestudyofat

In [11]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 234 texts...


Batches: 100%|██████████| 8/8 [00:10<00:00,  1.25s/it]


Generated embeddings with shape: (234, 384)
Adding 234 documents to vector store...
Successfully added 234 documents to vector store
Total documents in collection: 468


In [13]:
embeddings

array([[ 0.06025136, -0.11394929, -0.0458551 , ..., -0.03330351,
        -0.01776012, -0.01697645],
       [-0.0032879 , -0.10151993, -0.056716  , ...,  0.01702214,
         0.05128514, -0.03788602],
       [ 0.01857723, -0.10029167,  0.00346883, ...,  0.01033683,
         0.03356369, -0.03667551],
       ...,
       [-0.0147691 , -0.01492083, -0.03568733, ...,  0.04786457,
        -0.07905381, -0.0316795 ],
       [-0.05286935, -0.04104315,  0.0347242 , ...,  0.03187082,
        -0.07158925,  0.02121292],
       [-0.08416758,  0.03278182,  0.03379807, ...,  0.03418941,
        -0.12890713,  0.02126317]], shape=(234, 384), dtype=float32)

### Retriever Pipeline From VectorStore

In [14]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [16]:
rag_retriever

In [21]:
rag_retriever.retrieve("What is Object Detection Algorithms?")

Retrieving documents for query: 'What is Object Detection Algorithms?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 28.17it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_1ba4651f_198',
  'content': '4th International Conference on Innovative Academic Studies (ICIAS) – 12-13 March 2024 – Konya, Turkey \n \n392 \n \nIntroduction  \n \nObject detection is one of the computer vision tasks where that attempts to discover a specific object in a visual \nimage or video (K. LI & CAO, 2020). The goal of object detection is to develop a computational model or technique \nthat provides one of the simplest pieces of information needed by computer vision applications, which is one of \nthe fundamental problems of computer vision applications., What is there? And where is it located?  (Z. Zou, Shi, \nGuo, & Ye, 2019) . Detecting objects is one of the best -known challenges in computer vision (Krizhevsky, \nSutskever, & Hinton, 2017). Over the last decade, object discovery has become the most active and growing area \nof research in computer vision (Diwakar & Raj, 2022). Therefore, object detection, segmentation, object tracking,',
  'metadata': {'total_

### RAG Pipeline- VectorDB To LLM Output Generation

In [39]:
import os
from pathlib import Path
from dotenv import load_dotenv

env_candidates = [
    Path.cwd() / ".env",
    Path.cwd() / "RAG_PROJECT" / ".env",
    Path.cwd().parent / ".env",
    Path.cwd().parent.parent / ".env",
]
env_path = next((path for path in env_candidates if path.exists()), None)

if env_path is not None:
    load_dotenv(env_path)

print(f"GROQ_API_KEY configured: {bool(os.getenv('GROQ_API_KEY'))}")

GROQ_API_KEY configured: True


In [40]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [ ]:
class GroqLLM:
    def __init__(self, model_name: str = "qwen/qwen3.8-27b", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """Generate a response using retrieved context."""
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            response = self.llm.invoke([HumanMessage(content=formatted_prompt)])
            return response.content
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """Generate a simple response using retrieved context."""
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            response = self.llm.invoke([HumanMessage(content=simple_prompt)])
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"


In [42]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized Groq LLM with model: gemma2-9b-it
Groq LLM initialized successfully!


In [46]:
rag_retriever.retrieve("Object Detection Algorithms based on Deep Learning")

Retrieving documents for query: 'Object Detection Algorithms based on Deep Learning'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 26.55it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_21110ee9_207',
  'content': 'Jia, & Zhang, 2013). \n \nObject Detection Algorithms Based on Deep Learning  \n \nUndoubtedly, traditional ways of discovering things still work effectively, but deep learning in the field of \ncomputer vision has given a huge boost to development and is now a part of our daily lives. Neural-based methods \nhave added new features, increased accuracy and efficiency, and opened many ways to develop new technologies',
  'metadata': {'doc_index': 207,
   'producer': 'PDFium',
   'file_type': 'pdf',
   'source': '..\\data\\pdf\\objectdetection.pdf',
   'total_pages': 9,
   'page': 2,
   'page_label': '3',
   'content_length': 412,
   'creator': 'PDFium',
   'source_file': 'objectdetection.pdf',
   'creationdate': 'D:20240321141628'},
  'similarity_score': 0.5102766156196594,
  'distance': 0.4897233843803406,
  'rank': 1},
 {'id': 'doc_21cb4c21_207',
  'content': 'Jia, & Zhang, 2013). \n \nObject Detection Algorithms Based on Deep Learning  \n \nUn

### Integration Vectordb Context pipeline With LLM output

In [51]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="qwen/qwen3.8-27b",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
Context:
{context}

Question: {query}

Answer:"""
    
    response=llm.invoke(prompt)
    return response.content

In [53]:
answer=rag_simple("What is Object Detection Algorithms based on Deep Learning?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is Object Detection Algorithms based on Deep Learning?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 10.23it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Object detection algorithms based on deep learning are neural-based methods in computer vision that have significantly boosted development by adding new features, increasing accuracy and efficiency, and enabling new technologies for applications such as autonomous driving, human-computer interaction, intelligent video monitoring, and medical image sensing.


### Enhanced RAG Pipeline Features

In [54]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("What is Object Detection Algorithms based on Deep Learning?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What is Object Detection Algorithms based on Deep Learning?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 26.12it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: Object detection algorithms based on deep learning are neural-based methods in computer vision that have significantly boosted development by adding new features, increasing accuracy and efficiency, and enabling new technologies for applications such as autonomous driving, human-computer interaction, intelligent video monitoring, and medical image sensing.
Sources: [{'source': 'objectdetection.pdf', 'page': 2, 'score': 0.5525588989257812, 'preview': 'Jia, & Zhang, 2013). \n \nObject Detection Algorithms Based on Deep Learning  \n \nUndoubtedly, traditional ways of discovering things still work effectively, but deep learning in the field of \ncomputer vision has given a huge boost to development and is now a part of our daily lives. Neural-based metho...'}, {'source': 'objectdetection.pdf', 'page': 2, 'score': 0.5525588989257812, 'preview': 'Jia, & Zhang, 2013). \n \nObject Detection Algorithms Based on Deep Learning  \n \nUndoubtedly, traditional ways of discovering things stil